# Vet Clinics in Berlin – Cleaning & Normalization (v1)

This notebook takes the **v0** vet clinics dataset (OSM + LOR join) and:

1. Cleans and normalizes key fields needed for the product:
   - clinic name and full address
   - district and neighborhood
   - opening days and hours
   - contact information
   - basic accessibility and emergency service flags
   - latitude / longitude
2. Maps the data to a **target schema** aligned with the vet clinics modelling plan.
3. Exports a first **v1 cleaned vet clinics table** that can be integrated into the database or used by the app.

In [1]:
import pandas as pd
from pathlib import Path

# Path to v0 enriched dataset (OSM + LOR + lat/lon + basic attributes)
V0_PATH = Path("cache/vets_osm_berlin_with_lor_20251209_v0.csv")

df = pd.read_csv(V0_PATH)

print(f"Loaded v0 dataset from: {V0_PATH}")
print(f"Rows: {len(df)}, Columns: {df.shape[1]}")
df.head()## 1. Inspect input columns and basic structure

Loaded v0 dataset from: cache/vets_osm_berlin_with_lor_20251209_v0.csv
Rows: 175, Columns: 23


,id,@id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,contact:phone,website,...,opening_hours,operator,wheelchair,wheelchair:description,emergency,lor_id,district_name,neighborhood_name,lat,lon
0,way/24921047,way/24921047,Das Veterinärmedizinische Zentrum Berlin,Scharnweberstraße,136,13405.0,Berlin,+49 30 4127357,NaN,https://www.vetzentrum-berlin.de/,...,24/7,Kai S. Rödiger,NaN,NaN,NaN,re_ortsteil.1201,Reinickendorf,Reinickendorf,52.563836,13.325898
1,way/28608972,way/28608972,Zete Marton,Alt-Reinickendorf,37,13407.0,Berlin,NaN,NaN,NaN,...,"Mo-Fr 09:00-12:00, Mo,Th 16:00-19:00, Tu,We,Fr...",NaN,no,NaN,NaN,re_ortsteil.1201,Reinickendorf,Reinickendorf,52.574848,13.351279
2,way/71173694,way/71173694,Tierärztlichen Klinik für Kleintiere,Märkische Allee,258,12679.0,Berlin,NaN,NaN,https://www.tierklinik-in-berlin.de/,...,24/7,NaN,yes,NaN,NaN,re_ortsteil.1001,Marzahn-Hellersdorf,Marzahn,52.555555,13.553248
3,way/89101208,way/89101208,Tierarztpraxis Kathrin Böhm,Robert-Siewert-Straße,90,10318.0,Berlin,+49 30 32669736,NaN,NaN,...,"Th 10:00-18:00; Tu,We 10:00-19:00; Fr 10:00-15...",Kathrin Böhm,limited,NaN,NaN,re_ortsteil.1102,Lichtenberg,Karlshorst,52.492651,13.536917
4,way/117230559,way/117230559,Tierartzpraxis Gotthardt,Winkler Straße,21,14193.0,Berlin,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,re_ortsteil.0404,Charlottenburg-Wilmersdorf,Grunewald,52.487622,13.266606


## 1. Inspect input columns and basic structure

We first inspect the available columns in the v0 dataset to understand
which raw attributes we can map into the target schema.

In [2]:
df.columns.tolist()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175 entries, 0 to 174
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      175 non-null    object 
 1   @id                     175 non-null    object 
 2   name                    169 non-null    object 
 3   addr:street             127 non-null    object 
 4   addr:housenumber        127 non-null    object 
 5   addr:postcode           115 non-null    float64
 6   addr:city               113 non-null    object 
 7   phone                   72 non-null     object 
 8   contact:phone           25 non-null     object 
 9   website                 86 non-null     object 
 10  contact:website         24 non-null     object 
 11  email                   18 non-null     object 
 12  contact:email           12 non-null     object 
 13  opening_hours           125 non-null    object 
 14  operator                39 non-null     ob

## 2. Target schema for the cleaned vet clinics table

The cleaned vet clinics table should expose at least the following fields:

- `clinic_name` – Official or commonly used name of the vet clinic.
- `address` – Full street address (street + house number + postcode + city).
- `district` – Administrative Berlin district.
- `neighborhood` – Local neighborhood or Kiez (Ortsteil).
- `services_offered` – Free-text description or tags of services
  (e.g. general practice, surgery, emergency).
- `operating_days` – Days of operation (e.g. "Mon–Fri", "Mon–Sun").
- `operating_hours` – Opening hours in a human-readable format.
- `contact_info` – Aggregated contact field (phone, email, website).
- `latitude` / `longitude` – Coordinates for mapping.
- `accessibility_features` – Accessibility notes (e.g. wheelchair access).
- `data_source` – Original source of the data (for traceability).

The goal of this notebook is to build a `df_clean` DataFrame that follows
this schema as closely as possible using the available OSM attributes.

In [3]:
# 3. Build cleaned DataFrame aligned with the target schema

# Initialize an empty DataFrame with the same index as the input
df_clean = pd.DataFrame(index=df.index)

# 3.1 Clinic name
df_clean["clinic_name"] = df["name"].fillna("").astype(str).str.strip()

df_clean[["clinic_name"]].head()

,clinic_name
0,Das Veterinärmedizinische Zentrum Berlin
1,Zete Marton
2,Tierärztlichen Klinik für Kleintiere
3,Tierarztpraxis Kathrin Böhm
4,Tierartzpraxis Gotthardt


In [4]:
# 3.2 Full address (street + house number + postcode + city)

def build_full_address(row) -> str:
    """
    Compose a human-readable full address from OSM addr:* fields.

    Example format:
        "Musterstraße 12, 12345 Berlin"
    """
    parts = []

    street = row.get("addr:street", "")
    housenumber = row.get("addr:housenumber", "")
    postcode = row.get("addr:postcode", "")
    city = row.get("addr:city", "")

    # street + house number
    street_part = " ".join(
        p for p in [street, housenumber] if pd.notna(p) and str(p).strip()
    )
    if street_part:
        parts.append(street_part)

    # postcode + city
    city_part = " ".join(
        p for p in [str(postcode), city] if pd.notna(p) and str(p).strip()
    )
    if city_part:
        parts.append(city_part)

    return ", ".join(parts)


df_clean["address"] = df.apply(build_full_address, axis=1)

df_clean[["clinic_name", "address"]].head()

,clinic_name,address
0,Das Veterinärmedizinische Zentrum Berlin,"Scharnweberstraße 136, 13405.0 Berlin"
1,Zete Marton,"Alt-Reinickendorf 37, 13407.0 Berlin"
2,Tierärztlichen Klinik für Kleintiere,"Märkische Allee 258, 12679.0 Berlin"
3,Tierarztpraxis Kathrin Böhm,"Robert-Siewert-Straße 90, 10318.0 Berlin"
4,Tierartzpraxis Gotthardt,"Winkler Straße 21, 14193.0 Berlin"


In [5]:
# 3.3 District and neighborhood (Ortsteil)

df_clean["district"] = df["district_name"].fillna("").astype(str).str.strip()
df_clean["neighborhood"] = df["neighborhood_name"].fillna("").astype(str).str.strip()

df_clean[["district", "neighborhood"]].head()

,district,neighborhood
0,Reinickendorf,Reinickendorf
1,Reinickendorf,Reinickendorf
2,Marzahn-Hellersdorf,Marzahn
3,Lichtenberg,Karlshorst
4,Charlottenburg-Wilmersdorf,Grunewald


In [6]:
# 3.4 Latitude / longitude

df_clean["latitude"] = df["lat"]
df_clean["longitude"] = df["lon"]

df_clean[["latitude", "longitude"]].head()

,latitude,longitude
0,52.563836,13.325898
1,52.574848,13.351279
2,52.555555,13.553248
3,52.492651,13.536917
4,52.487622,13.266606


In [7]:
# 3.5 Aggregated contact info (phone, email, website)

def build_contact_info(row) -> str:
    """
    Aggregate available contact fields into a single string.
    Fields used:
      - phone, contact:phone
      - email, contact:email
      - website, contact:website
    """
    pieces = []

    phones = []
    emails = []
    websites = []

    # Phones
    for col in ["phone", "contact:phone"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            phones.append(str(row[col]).strip())

    # Emails
    for col in ["email", "contact:email"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            emails.append(str(row[col]).strip())

    # Websites
    for col in ["website", "contact:website"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            websites.append(str(row[col]).strip())

    if phones:
        pieces.append("Phone: " + " / ".join(sorted(set(phones))))
    if emails:
        pieces.append("Email: " + " / ".join(sorted(set(emails))))
    if websites:
        pieces.append("Website: " + " / ".join(sorted(set(websites))))

    return " | ".join(pieces)


df_clean["contact_info"] = df.apply(build_contact_info, axis=1)

df_clean[["clinic_name", "contact_info"]].head()

,clinic_name,contact_info
0,Das Veterinärmedizinische Zentrum Berlin,Phone: +49 30 4127357 | Email: info@vetzentrum...
1,Zete Marton,
2,Tierärztlichen Klinik für Kleintiere,Website: https://www.tierklinik-in-berlin.de/
3,Tierarztpraxis Kathrin Böhm,Phone: +49 30 32669736
4,Tierartzpraxis Gotthardt,


In [8]:
# 3.6 Operating hours and operating days
# We use OSM 'opening_hours' as-is for operating_hours
# and derive a simple operating_days label using heuristics.

df_clean["operating_hours"] = (
    df["opening_hours"].fillna("").astype(str).str.strip()
)


def infer_operating_days(opening_hours_str: str) -> str:
    """
    Very simple heuristic to derive operating_days from opening_hours.

    This is intentionally conservative and should not be over-interpreted.
    """
    if not opening_hours_str:
        return ""

    text = opening_hours_str.lower()
    no_space = text.replace(" ", "")

    if "24/7" in text:
        return "Mon–Sun"
    if "mo-su" in no_space or "mo-su" in text:
        return "Mon–Sun"
    if "mo-fr" in no_space or "mo-fr" in text:
        return "Mon–Fri"

    # Fallback: unknown / not derived
    return ""


df_clean["operating_days"] = df_clean["operating_hours"].apply(infer_operating_days)

df_clean[["operating_days", "operating_hours"]].head()

,operating_days,operating_hours
0,Mon–Sun,24/7
1,Mon–Fri,"Mo-Fr 09:00-12:00, Mo,Th 16:00-19:00, Tu,We,Fr..."
2,Mon–Sun,24/7
3,,"Th 10:00-18:00; Tu,We 10:00-19:00; Fr 10:00-15..."
4,,


In [9]:
# 3.7 Services offered
# For v1, we only use the 'emergency' flag from OSM, if present.
# Additional enrichment (e.g. specialities from websites) can be added later.

def build_services_offered(row) -> str:
    services = []

    emergency_val = row.get("emergency", "")
    if pd.notna(emergency_val):
        val = str(emergency_val).strip().lower()
        if val in ["yes", "true", "24/7", "24-7", "24h"]:
            services.append("emergency")

    # We deliberately do not infer "general practice" for all clinics here.
    return ", ".join(services)


df_clean["services_offered"] = df.apply(build_services_offered, axis=1)

df_clean[["clinic_name", "services_offered"]].head()

,clinic_name,services_offered
0,Das Veterinärmedizinische Zentrum Berlin,
1,Zete Marton,
2,Tierärztlichen Klinik für Kleintiere,
3,Tierarztpraxis Kathrin Böhm,
4,Tierartzpraxis Gotthardt,


In [10]:
# 3.8 Accessibility features (based on wheelchair tags)

def build_accessibility_features(row) -> str:
    """
    Combine wheelchair and wheelchair:description into a single text field.
    """
    parts = []

    wc = row.get("wheelchair", "")
    wc_desc = row.get("wheelchair:description", "")

    if pd.notna(wc) and str(wc).strip():
        parts.append(f"wheelchair={str(wc).strip()}")

    if pd.notna(wc_desc) and str(wc_desc).strip():
        parts.append(str(wc_desc).strip())

    return " | ".join(parts)


df_clean["accessibility_features"] = df.apply(build_accessibility_features, axis=1)

df_clean[["clinic_name", "accessibility_features"]].head()

,clinic_name,accessibility_features
0,Das Veterinärmedizinische Zentrum Berlin,
1,Zete Marton,wheelchair=no
2,Tierärztlichen Klinik für Kleintiere,wheelchair=yes
3,Tierarztpraxis Kathrin Böhm,wheelchair=limited
4,Tierartzpraxis Gotthardt,


In [11]:
# 3.9 Data source (for traceability)

df_clean["data_source"] = "OSM amenity=veterinary, Berlin, snapshot 2025-12-09 (Overpass)"

df_clean[["clinic_name", "data_source"]].head()

,clinic_name,data_source
0,Das Veterinärmedizinische Zentrum Berlin,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
1,Zete Marton,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
2,Tierärztlichen Klinik für Kleintiere,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
3,Tierarztpraxis Kathrin Böhm,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
4,Tierartzpraxis Gotthardt,"OSM amenity=veterinary, Berlin, snapshot 2025-..."


## 3. Handling missing clinic names

Some OSM features do not have a `name` tag. To avoid empty `clinic_name`
values in the final table, we apply a simple fallback strategy:

1. Use `clinic_name` if present.
2. Else, try the `operator` field.
3. Else, fall back to `"Veterinary clinic - <address>"` when an address exists.
4. As a last resort, use a generic label `"Veterinary clinic (name missing)"`.

In [12]:
def ensure_clinic_name(row):
    """
    Ensure clinic_name is not empty by applying a fallback strategy:
    - If clinic_name is present: keep it.
    - Else if operator is present: use operator as clinic_name.
    - Else use a generic label based on the address.
    """
    # 1) If we already have a non-empty clinic_name, keep it
    if row["clinic_name"]:
        return row["clinic_name"]

    # 2) Try to use 'operator' from the original v0 DataFrame
    op = None
    if "operator" in df.columns:
        op = df.loc[row.name, "operator"]
        if pd.notna(op) and str(op).strip():
            return str(op).strip()

    # 3) Fallback: use the address if available
    address = row.get("address", "")
    if address:
        return f"Veterinary clinic - {address}"

    # 4) Final fallback: generic label
    return "Veterinary clinic (name missing)"


# Apply fallback to all rows in df_clean
df_clean["clinic_name"] = df_clean.apply(ensure_clinic_name, axis=1)

# Check rows that were previously empty
missing_name_mask = df["name"].isna() | (df["name"].astype(str).str.strip() == "")
df_clean.loc[missing_name_mask, ["clinic_name", "address", "district", "neighborhood"]].head(10)

,clinic_name,address,district,neighborhood
27,Veterinary clinic - nan,nan,Charlottenburg-Wilmersdorf,Schmargendorf
30,"Veterinary clinic - Bruno-Baum-Straße 75, 1268...","Bruno-Baum-Straße 75, 12685.0 Berlin",Marzahn-Hellersdorf,Marzahn
47,Veterinary clinic - nan,nan,Spandau,Spandau
84,"Veterinary clinic - Wattstraße 10, 13629.0","Wattstraße 10, 13629.0",Spandau,Siemensstadt
114,Veterinary clinic - nan,nan,Pankow,Wilhelmsruh
148,Veterinary clinic - nan,nan,Marzahn-Hellersdorf,Mahlsdorf


## 4. Final schema check and basic quality checks

We now:

1. Reorder columns to match the target schema.
2. Check for missing values and empty strings in key fields.

In [13]:
target_cols = [
    "clinic_name",
    "address",
    "district",
    "neighborhood",
    "services_offered",
    "operating_days",
    "operating_hours",
    "contact_info",
    "latitude",
    "longitude",
    "accessibility_features",
    "data_source",
]

# Ensure all expected columns exist
missing_cols = [c for c in target_cols if c not in df_clean.columns]
print("Missing columns (should be empty):", missing_cols)

df_clean = df_clean[target_cols]

print("\nNull counts per column:")
print(df_clean.isna().sum())

print("\nEmpty clinic_name rows:", (df_clean["clinic_name"] == "").sum())
print("Empty address rows:", (df_clean["address"] == "").sum())
print("Empty district rows:", (df_clean["district"] == "").sum())
print("Empty neighborhood rows:", (df_clean["neighborhood"] == "").sum())

df_clean.head()

Missing columns (should be empty): []

Null counts per column:
clinic_name               0
address                   0
district                  0
neighborhood              0
services_offered          0
operating_days            0
operating_hours           0
contact_info              0
latitude                  0
longitude                 0
accessibility_features    0
data_source               0
dtype: int64

Empty clinic_name rows: 0
Empty address rows: 0
Empty district rows: 0
Empty neighborhood rows: 0


,clinic_name,address,district,neighborhood,services_offered,operating_days,operating_hours,contact_info,latitude,longitude,accessibility_features,data_source
0,Das Veterinärmedizinische Zentrum Berlin,"Scharnweberstraße 136, 13405.0 Berlin",Reinickendorf,Reinickendorf,,Mon–Sun,24/7,Phone: +49 30 4127357 | Email: info@vetzentrum...,52.563836,13.325898,,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
1,Zete Marton,"Alt-Reinickendorf 37, 13407.0 Berlin",Reinickendorf,Reinickendorf,,Mon–Fri,"Mo-Fr 09:00-12:00, Mo,Th 16:00-19:00, Tu,We,Fr...",,52.574848,13.351279,wheelchair=no,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
2,Tierärztlichen Klinik für Kleintiere,"Märkische Allee 258, 12679.0 Berlin",Marzahn-Hellersdorf,Marzahn,,Mon–Sun,24/7,Website: https://www.tierklinik-in-berlin.de/,52.555555,13.553248,wheelchair=yes,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
3,Tierarztpraxis Kathrin Böhm,"Robert-Siewert-Straße 90, 10318.0 Berlin",Lichtenberg,Karlshorst,,,"Th 10:00-18:00; Tu,We 10:00-19:00; Fr 10:00-15...",Phone: +49 30 32669736,52.492651,13.536917,wheelchair=limited,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
4,Tierartzpraxis Gotthardt,"Winkler Straße 21, 14193.0 Berlin",Charlottenburg-Wilmersdorf,Grunewald,,,,,52.487622,13.266606,,"OSM amenity=veterinary, Berlin, snapshot 2025-..."


## 5. Export cleaned vet clinics dataset (v1)

We export the cleaned vet clinics table to a CSV file in `cache/`:

- `vet_clinics_berlin_clean_20251209_v1.csv`

This file follows the target schema and can be used as input for
the database integration or downstream analysis.

In [14]:
OUTPUT_PATH = Path("cache/vet_clinics_berlin_clean_20251209_v1.csv")

df_clean.to_csv(OUTPUT_PATH, index=False)

print(f"Exported cleaned vet clinics dataset to: {OUTPUT_PATH}")
df_clean.head()

Exported cleaned vet clinics dataset to: cache/vet_clinics_berlin_clean_20251209_v1.csv


,clinic_name,address,district,neighborhood,services_offered,operating_days,operating_hours,contact_info,latitude,longitude,accessibility_features,data_source
0,Das Veterinärmedizinische Zentrum Berlin,"Scharnweberstraße 136, 13405.0 Berlin",Reinickendorf,Reinickendorf,,Mon–Sun,24/7,Phone: +49 30 4127357 | Email: info@vetzentrum...,52.563836,13.325898,,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
1,Zete Marton,"Alt-Reinickendorf 37, 13407.0 Berlin",Reinickendorf,Reinickendorf,,Mon–Fri,"Mo-Fr 09:00-12:00, Mo,Th 16:00-19:00, Tu,We,Fr...",,52.574848,13.351279,wheelchair=no,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
2,Tierärztlichen Klinik für Kleintiere,"Märkische Allee 258, 12679.0 Berlin",Marzahn-Hellersdorf,Marzahn,,Mon–Sun,24/7,Website: https://www.tierklinik-in-berlin.de/,52.555555,13.553248,wheelchair=yes,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
3,Tierarztpraxis Kathrin Böhm,"Robert-Siewert-Straße 90, 10318.0 Berlin",Lichtenberg,Karlshorst,,,"Th 10:00-18:00; Tu,We 10:00-19:00; Fr 10:00-15...",Phone: +49 30 32669736,52.492651,13.536917,wheelchair=limited,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
4,Tierartzpraxis Gotthardt,"Winkler Straße 21, 14193.0 Berlin",Charlottenburg-Wilmersdorf,Grunewald,,,,,52.487622,13.266606,,"OSM amenity=veterinary, Berlin, snapshot 2025-..."
